# 4.3 FA 算子工程介绍：目录结构与三层映射

官方 FA 算子源码位于 [ops-transformer 仓库](https://gitcode.com/cann/ops-transformer/tree/master/attention/fused_infer_attention_score)。本节把工程目录过一遍，并把每个位置映射回第 3 章的三层结构。

## 1. 工程目录总览

```text
fused_infer_attention_score/
├── CMakeLists.txt                 # 构建入口
├── op_host/                       # 【Host 侧】
│   ├── fused_infer_attention_score_tiling.h     # TilingData 结构定义
│   ├── fused_infer_attention_score_tiling.cpp   # TilingFunc 实现（多模板）
│   └── fused_infer_attention_score.cpp          # 算子 IR 原型注册（属性/输入输出声明）
├── op_kernel/                     # 【Device 侧】
│   ├── fused_infer_attention_score.cc           # Kernel 主入口
│   ├── fused_infer_attention_score_common.h     # 公共宏/工具
│   ├── kernel_fia_*.h             # 按场景隔离的 Kernel 实现（配合 TilingKey）
│   └── ...
├── tests/                         # 调测
│   ├── debug_tools/               # DebugTools 调测工程（4.5 节主角）
│   └── golden/                    # 标准答案生成脚本
└── info.json                      # 算子元信息
```

> 工程的物理划分（op_host / op_kernel）正是第 3 章逻辑分层（Tiling / Kernel）的代码落点。

## 2. op_host：算子原型与 Tiling

| 文件 | 职责 | 对应第 3 章概念 |
|--|--|--|
| `*_tiling.h` | 定义 `TilingData` 结构：inputParams / coreParams / multiCoreParams / bmm1TilingData / bmm2TilingData / softmaxFlashTilingData | 3.5 节各步骤的「表格」 |
| `*_tiling.cpp` | 实现 `DoTiling` 框架与各模板：GetShapeAttrsInfo → ... → SetTilingKey | 3.4 / 3.5 节八步流程 |
| `*.cpp`（IR 原型） | 注册算子名、属性（numHeads、scaleValue、inputLayout…）、输入输出与排布约束 | 3.2 节参数表 |

## 3. op_kernel：核函数

| 文件 | 职责 | 对应第 3 章概念 |
|--|--|--|
| `*.cc` 主入口 | `extern "C" __global__ __aicore__` 核函数，读 TilingData、SPMD 分任务 | 3.6 节 Kernel 框架 |
| `kernel_fia_*.h` | 按 TilingKey 编译期隔离的场景实现（不同 layout / mask / 尾块组合） | 3.4 节 TilingKey 隔离 |

> **分离架构的编译产物**：kernel 编译后会生成 **AIC 和 AIV 两个 `.o`**（详见 4.5 节调测流程）——Cube 核与 Vector 核代码分别编译、按组合启动。

## 4. 构建方式

ops-transformer 使用 CMake 统一构建：

```bash
# 克隆仓库并切到目标分支
git clone https://gitcode.com/cann/ops-transformer.git
cd ops-transformer

# 编译（产物在 build/ 下，含 aic/aiv 两个 .o 与 host 库）
bash build.sh

# 单算子直调调测工程见 tests/debug_tools/
```

构建产物的对应关系：

| 产物 | 内容 | 运行位置 |
|--|--|--|
| host 侧库 | 算子原型注册 + TilingFunc | CPU |
| `*_aic.o` | Cube 核代码 | AIC |
| `*_aiv.o` | Vector 核代码 | AIV |

## 4. 三层结构回顾（对号入座）

```text
调用通路（3.2）：aclnnFusedInferAttentionScoreGetWorkspaceSize / 执行
   └─ op_host/*.cpp 的 IR 原型注册被算子执行框架加载
Tiling（3.4/3.5）：op_host/*_tiling.cpp 多模板
   └─ 产出 TilingData + TilingKey + blockDim
Kernel（3.6/3.7）：op_kernel/*.cc + kernel_fia_*.h
   └─ 核间切块 + 四拍循环 + 乒乓 / preload
```

## 小测验

1. `op_host` 和 `op_kernel` 分别对应第 3 章的哪两层？各自跑在什么处理器上？
2. Kernel 编译为什么产出两个 `.o`？对应哪两个单元？
3. `kernel_fia_*.h` 按场景拆分成多个文件的动机是什么？（提示：第 3 章 3.4 节原则二）